<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# CP101: Lab 11 - Accessibility & Publishing the Interactive Map

By the end of this lab, you will be able to:

1. Determine the number of amenities within a walking catchment.
2. Make interactive maps with pydeck and lonboard, and know when to choose one vs the other.
3. Export an interactive map to a standalone `.html` file.


In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("CP101_LAB11.ipynb")

In [ ]:
%pip install -q -U "anywidget>=0.11,<0.12" lonboard pydeck

In [ ]:
import os
import time
import geopandas as gpd
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import pydeck as pdk
import shapely
from shapely.geometry import LineString, box
from lonboard import Map, PolygonLayer, ScatterplotLayer
from lonboard.layer_extension import DataFilterExtension

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Overview

A simple way to measure accessibility is to draw a **catchment** (the area reachable within a set walking distance) around a location, then count the **amenities** (preschools, cafes, pharmacies, etc.) inside of it. 

We will use a 10-minute walk, measured along the real street network.

Two libraries make interactive maps from Python. Both are built on [deck.gl](https://deck.gl/) and both export a standalone `.html` file:

| | [pydeck](https://deckgl.readthedocs.io/en/latest/) | [lonboard](https://developmentseed.org/lonboard/latest/) |
|---|---|---|
| Data sent to the map | JSON | Parquet binary, straight to the GPU |
| Best for | Small to medium datasets, many layer types | Large datasets from GeoPandas |
| Export | `Deck.to_html("map.html")` | `Map.to_html("map.html")` |

### Load the data

`places` is every place in San Francisco from [Overture Maps](https://overturemaps.org/).

In [ ]:
places = gpd.read_parquet("data/sf_places_all.parquet").set_crs(4326, allow_override=True)

CRS_M = 26910 # a CRS in meters, used for the grid
WALK_SPEED = 80 # meters per minute (about 4.8 km/h)
WALK_MIN = 10 # minutes

PALETTE = [[34, 139, 34], [0, 170, 120], [139, 90, 43], [230, 90, 40], [200, 30, 30], [70, 70, 200]]    # map colors

### What kinds of places are there?

Every place has a `category`, such as `restaurant` or `pharmacy`. How many are there, and which are most common?

In [ ]:
print(len(places), "places in", places["category"].nunique(), "categories")
places["category"].value_counts().head(40)

### ✏️ Your turn: choose your categories

From the list of available categories, pick **6 types** you'd count as amenities and add them to the `keep` list, using the exact names from the table above. `preschool` is already in the list because we will use it in the coming sections, so make sure to keep it in your category list.

In [ ]:
# replace each ... with another category name
keep = [
    "preschool",
    "...",
    "...",
    "...",
    "...",
    "...",
]

In [ ]:
grader.check("q1_categories")

### Build the amenities table

This keeps only places in your chosen categories with a `confidence` above 0.7 (Overture's own score for how likely a place is real), and gives each category a map color.

In [ ]:
amenities = places[places["category"].isin(keep) & (places["confidence"] > 0.7)]
amenities = amenities[["name", "category", "geometry"]].reset_index(drop=True)

colors = dict(zip(keep, PALETTE))    # one map color per type
amenities["category"].value_counts()

In [ ]:
# small helper: print a color key under a map
from IPython.display import HTML, display

def legend(items):
    """items = [(label, [r, g, b]), ...]"""
    dots = "".join(
        f'<span style="margin-right:14px"><span style="display:inline-block;width:12px;height:12px;'
        f'border-radius:50%;margin-right:4px;background:rgb({int(c[0])},{int(c[1])},{int(c[2])})"></span>{label}</span>'
        for label, c in items
    )
    display(HTML(f'<div style="font-size:13px;margin:4px 0">{dots}</div>'))

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part 1: Amenities within walking catchments

We'll compare three BART stations: Civic Center, 16th St Mission, and Powell St. Our **walking catchment** is everything you can reach in 10 minutes **along the street network**, not a circle on the map. Each step adds a layer to the map at the start of Part 2:

1. Load the walking street network from OpenStreetMap.
2. Snap each station to its nearest network node.
3. Find every node reachable within 10 minutes, and wrap them in a catchment polygon.
4. Count the amenities inside each catchment.

### Step 1: The walking network

In the street graph, each edge has a `length` in meters. We add `walk_min`, the minutes to walk it, which is calculated by taking the `length` divided by the `WALK_SPEED`.

In [ ]:
nodes = gpd.read_parquet("data/sf_walk_nodes.parquet")
edges = gpd.read_parquet("data/sf_walk_edges.parquet")

G = ox.graph_from_gdfs(nodes, edges)
for u, v, k, d in G.edges(keys=True, data=True):
    d["walk_min"] = d["length"] / WALK_SPEED
    
print(len(G.nodes), "nodes,", len(G.edges), "edges")

### Step 2: Snap the stations to the network

In [ ]:
stations = gpd.GeoDataFrame(
    {"station": ["Civic Center", "16th St Mission", "Powell St"]},
    geometry=gpd.points_from_xy([-122.4142, -122.4196, -122.4079], [37.7794, 37.7650, 37.7845]),
    crs=4326,
)
stations["node"] = ox.distance.nearest_nodes(G, stations.geometry.x, stations.geometry.y)
stations

### Step 3: Walking catchments

`nx.ego_graph` keeps every node within `WALK_MIN` minutes of the station. A concave hull around those nodes is the catchment.

In [ ]:
polys, edge_parts = [], []
for n in stations["node"]:
    # every street node within WALK_MIN minutes of the station
    reach = nx.ego_graph(G, n, radius=WALK_MIN, distance="walk_min")
    reach_nodes, reach_edges = ox.graph_to_gdfs(reach)

    # wrap those nodes in a polygon: the catchment
    polys.append(shapely.concave_hull(reach_nodes.union_all(), ratio=0.3))
    # keep the streets too, to draw on the map
    edge_parts.append(reach_edges[["geometry"]])

catchments = gpd.GeoDataFrame(stations[["station"]], geometry=polys, crs=4326)
walk_edges = pd.concat(edge_parts)
catchments

### Step 4: Count the amenities inside

A **spatial join** (`gpd.sjoin`) attaches each catchment's columns to the amenities that fall inside it. Grouping the result by station and category then gives the counts.

### ✏️ Your turn: count the amenities

Replace the `...` below to complete the spatial join for `amenities` within `catchments`.

In [ ]:
hits = gpd.sjoin(...)    # complete the spatial join

counts = hits.groupby(["station", "category"]).size().unstack(fill_value=0)    # group by station and category and count occurrences
counts

In [ ]:
grader.check("q2_catchments")

### ✏️ Try it Out: compare the stations

Which station has the most amenities of any kind within walking distance? Which type of amenity is most common near each station?

*Optional:* change `WALK_MIN` to `5` in the setup cell and rerun from there to see how the counts change. Set it back to `10` when you're done.

*Your answer here.*

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part 2: Mapping with pydeck

[pydeck](https://deckgl.readthedocs.io/en/latest/) is the Python interface to deck.gl. You build a map from **layers**: each layer has a type (such as `GeoJsonLayer` for polygons, `ScatterplotLayer` for points), a dataset, and styling. pydeck converts your data to **JSON** and sends it to the browser, which is simple and works with many layer types, but gets slow for very large datasets.

pydeck needs coordinates in plain `lon` and `lat` columns.

### Step 1: Prepare the data

pydeck can draw a GeoDataFrame of lines or polygons directly (`walk_edges`, `catchments`). Points need plain `lon` and `lat` columns, so we add those for the amenities and the stations.

In [ ]:
# amenities inside the catchments
inside = hits.copy()
inside["lon"] = inside.geometry.x                   # plain lon and lat columns for pydeck
inside["lat"] = inside.geometry.y
inside["color"] = inside["category"].map(colors)    # one color per amenity type
inside = inside[["lon", "lat", "name", "category", "color"]]    # keep only what the map uses

# stations: same steps, with station renamed to name so we can add a 'name' tooltip in the map
stn = stations.rename(columns={"station": "name"}).copy()
stn["lon"] = stn.geometry.x
stn["lat"] = stn.geometry.y
stn = stn[["lon", "lat", "name"]]

### Step 2: Make one layer per step

Each pydeck layer `pdk.Layer` takes a layer type, the data, and styling. Colors are given as RGB values, with an optional fourth number for transparency. `get_radius` is in meters, and `pickable=True` shows a tooltip when you hover.

In [ ]:
# lines and polygons: pass the GeoDataFrame directly
network_layer = pdk.Layer("GeoJsonLayer", walk_edges, get_line_color=[150, 150, 150], line_width_min_pixels=1)
catch_layer = pdk.Layer("GeoJsonLayer", catchments, get_fill_color=[30, 130, 200, 50], get_line_color=[30, 130, 200])

# points: say which columns hold the position and the color
amenity_layer = pdk.Layer(
    "ScatterplotLayer", inside,
    get_position=["lon", "lat"], get_fill_color="color", get_radius=20, pickable=True,
)
station_layer = pdk.Layer(
    "ScatterplotLayer", stn,
    get_position=["lon", "lat"], get_fill_color=[255, 215, 0], get_line_color=[0, 0, 0],
    stroked=True, line_width_min_pixels=2, get_radius=60, pickable=True,
)

### Step 3: Stack the layers into a map

`pdk.Deck` draws the layers in the order you list them, bottom to top. `ViewState` sets where the map starts.

In [ ]:
catchment_deck = pdk.Deck(
    layers=[network_layer, catch_layer, amenity_layer, station_layer],
    initial_view_state=pdk.ViewState(latitude=37.775, longitude=-122.415, zoom=13),
    tooltip={"text": "{name}"},
)
legend([(c, colors[c]) for c in keep] + [("station", [255, 215, 0])])
catchment_deck

Each layer is one step from Part 1, stacked bottom to top:

- **Gold dots:** the stations (Step 2).
- **Gray lines:** the walkable streets inside each catchment (Step 3).
- **Blue areas:** the walking catchments (Step 3).
- **Colored dots:** amenities inside them, by type (Step 4).


### ✏️ Your turn: highlight the preschools

On the map above, every amenity type is a small dot. Make all the preschools in San Francisco stand out by building a new pydeck layer for them. Replace the three `...` below:

- the **layer type**: a string, like the `"ScatterplotLayer"` used in the cells above;
- the **fill color**: `[red, green, blue]`, each from 0 to 255;
- the **radius** in meters: bigger than the 20 m amenity dots, between 40 and 300.

In [ ]:
preschools = amenities[amenities["category"] == "preschool"].copy()
preschools["lon"] = preschools.geometry.x
preschools["lat"] = preschools.geometry.y
preschools = preschools[["lon", "lat", "name"]]

preschool_layer = pdk.Layer(
    ...,                         # layer type, as a string
    preschools,
    get_position=["lon", "lat"],
    get_fill_color=...,          # [r, g, b]
    get_radius=...,              # in meters
    pickable=True,
)

preschool_deck = pdk.Deck(
    layers=[*catchment_deck.layers, preschool_layer],
    initial_view_state=catchment_deck.initial_view_state,
    tooltip={"text": "{name}"},
)
legend([("preschool", preschool_layer.get_fill_color)])
preschool_deck

In [ ]:
grader.check("q3_pydeck")

### Where is access missing?

Now flip the question: which parts of San Francisco are **more than a 10 minute walk from any preschool**? We'll answer it by breaking the city into 250m squares and then find the walking time to the nearest preschool for each.

1. Cover the city with a grid of 250 m squares.
2. Snap each square's center to its nearest street node.
3. Find the walking time from each node to the nearest preschool.

In [ ]:
# a grid of 250 m squares covering the street network
nodes_m = nodes.to_crs(CRS_M)
cell = 250
minx, miny, maxx, maxy = nodes_m.total_bounds
grid = gpd.GeoDataFrame(
    geometry=[box(x, y, x + cell, y + cell)
              for x in np.arange(minx, maxx, cell) for y in np.arange(miny, maxy, cell)],
    crs=CRS_M,
)

# drop squares with no streets in them (water, mostly)
has_streets = gpd.sjoin(grid, nodes_m[["geometry"]]).index
grid = grid[grid.index.isin(has_streets)].reset_index(drop=True)

# snap each square's center to its nearest street node
centers = grid.centroid.to_crs(4326)
grid["node"] = ox.distance.nearest_nodes(G, centers.x, centers.y)
print(len(grid), "cells")

`walk_minutes(category)` is a helper we wrote for you: give it an amenity type and it returns the walking minutes from every grid square to the nearest one. It searches outward from all places of that type at once, which is much faster than routing from each square separately.

In [ ]:
def walk_minutes(category):
    """Walking minutes from each grid cell to the nearest place of this category."""
    # the street node nearest to each place of this type
    pts = amenities[amenities["category"] == category]
    sources = set(ox.distance.nearest_nodes(G, pts.geometry.x, pts.geometry.y))

    # walking minutes from the nearest source to every street node
    to_node = nx.multi_source_dijkstra_path_length(G, sources, weight="walk_min")

    # look up each cell's node (cells that can't reach a place get infinity)
    return grid["node"].map(to_node).fillna(np.inf)

Run it for one type, then color each square: light blue if a preschool is within `WALK_MIN` minutes, red if not.

In [ ]:
kind = "preschool"

# walking minutes from each cell to the nearest preschool
grid["minutes"] = walk_minutes(kind)

# light blue: a preschool within WALK_MIN minutes. red: farther than that (a gap)
grid["color"] = [[200, 30, 30, 100] if m > WALK_MIN else [120, 180, 230, 100] for m in grid["minutes"]]
print(f"{(grid['minutes'] > WALK_MIN).mean():.0%} of San Francisco is more than {WALK_MIN} minutes from a {kind}")

Find the cell farthest from any preschool, and the walking route to it from the nearest one.

In [ ]:
# the worst gap: the cell farthest from any preschool
worst = grid.loc[grid["minutes"].replace(np.inf, np.nan).idxmax()]

# search again from all preschools at once, this time keeping the paths
pts = amenities[amenities["category"] == kind]
sources = set(ox.distance.nearest_nodes(G, pts.geometry.x, pts.geometry.y))
minutes, paths = nx.multi_source_dijkstra(G, sources, weight="walk_min")

# the walking route from the nearest preschool to that cell
line = LineString(nodes.loc[paths[worst["node"]], "geometry"].tolist())
route = gpd.GeoDataFrame({"minutes": [round(minutes[worst["node"]], 1)]}, geometry=[line], crs=4326)
print(f"The farthest cell is a {route['minutes'][0]} minute walk from a {kind}")

Now draw it with pydeck: the grid squares, the walking route, and the preschools.

In [ ]:
# the preschools, as points with lon and lat columns
found_pts = amenities[amenities["category"] == kind].copy()
found_pts["lon"] = found_pts.geometry.x
found_pts["lat"] = found_pts.geometry.y
found_pts = found_pts[["lon", "lat", "name"]]

# every grid cell, colored light blue or red
cell_layer = pdk.Layer(
    "GeoJsonLayer", grid.to_crs(4326)[["color", "geometry"]],
    get_fill_color="color", get_line_color=[255, 255, 255, 80], line_width_min_pixels=1,
)
route_layer = pdk.Layer("GeoJsonLayer", route, get_line_color=[0, 255, 0], line_width_min_pixels=4)
found_layer = pdk.Layer(
    "ScatterplotLayer", found_pts,
    get_position=["lon", "lat"], get_fill_color=[255,255,0], get_radius=40, pickable=True,
)
gaps_deck = pdk.Deck(
    layers=[cell_layer, route_layer, found_layer],
    initial_view_state=pdk.ViewState(latitude=37.76, longitude=-122.44, zoom=11),
    tooltip={"text": "{name}"},
)
gaps_deck

<span style="display:inline-block;width:14px;height:14px;background:rgba(120,180,230,0.55);border:1px solid #888;"></span> within a 10-minute walk of a preschool &nbsp;&nbsp;
<span style="display:inline-block;width:14px;height:14px;background:rgba(200,30,30,0.55);border:1px solid #888;"></span> more than 10 minutes (a gap) &nbsp;&nbsp;
<span style="display:inline-block;width:14px;height:14px;background:#ffff00;border-radius:50%;"></span> preschool &nbsp;&nbsp;
<span style="display:inline-block;width:20px;height:4px;background:#0f0;vertical-align:middle;"></span> walking route from the farthest cell

*Try it:* change `kind` to another type in your `keep` list and rerun the cells above, down to the map.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part 3: Mapping with lonboard

[lonboard](https://developmentseed.org/lonboard/latest/) skips the JSON step. It takes a GeoDataFrame directly and sends the geometry to the browser in binary, which the GPU can read without conversion. That makes it much faster for large datasets, and it adds GPU features such as filtering. We don't need `lon` and `lat` columns or a reprojection to build the layer, as long as the GeoDataFrame has a CRS.

### Amenities, with a filter

`DataFilterExtension` filters points on the GPU, so toggling a type is instant even for big data. It needs numeric codes, so we turn each category into an integer.

In [ ]:
# the filter works on integer codes, so give each category a number
cats = amenities["category"].astype("category")
codes = {name: i for i, name in enumerate(cats.cat.categories)}

show = keep    # the types to show (all six to start)

amenity_map_layer = ScatterplotLayer.from_geopandas(
    amenities,                                                                       # the GeoDataFrame, as is
    get_fill_color=np.array(amenities["category"].map(colors).tolist(), dtype="uint8"),    # one [r, g, b] per place
    radius_min_pixels=3,
    extensions=[DataFilterExtension(filter_size=None, category_size=1)],             # turn on category filtering
    get_filter_category=cats.cat.codes.to_numpy(),                                   # each place's category code
    filter_categories=[codes[c] for c in show],                                      # only show these codes
)
filter_map = Map(amenity_map_layer)
legend([(c, colors[c]) for c in show])
filter_map

*Try it:* change `show` to just one or two types from `keep`, like `show = ["preschool"]`, then rerun the cell above. Only those types stay on the map.

### ✏️ Your turn: a lonboard layer for the preschools

lonboard takes a GeoDataFrame directly, with no `lon` and `lat` columns. Build a layer for just the preschools. Replace the three `...` below:

- the **data**: the `preschool_points` GeoDataFrame;
- the **fill color**: `[red, green, blue]`, each from 0 to 255;
- the **radius** in pixels (`radius_min_pixels`): between 3 and 12.

In [ ]:
preschool_points = amenities[amenities["category"] == "preschool"]

preschool_lb = ScatterplotLayer.from_geopandas(
    ...,                         # the GeoDataFrame
    get_fill_color=...,          # [r, g, b]
    radius_min_pixels=...,       # in pixels
)

# show the layer on a map
legend([("preschool", preschool_lb.get_fill_color)])
Map(preschool_lb)

In [ ]:
grader.check("q4_lonboard")

### Accessibility surface

Now the citywide view. Using the same grid, count how many of your six amenity types are within a 10-minute walk of each cell.

In [ ]:
# for each type, is the nearest one within WALK_MIN minutes (True/False)? Add them up across the types
grid["n_types"] = sum(walk_minutes(c) <= WALK_MIN for c in keep).astype(int)
grid["n_types"].value_counts().sort_index()

In [ ]:
# light to dark green: 0 types to 6 types
ramp = np.array([
    [247, 252, 245], [229, 245, 224], [199, 233, 192], [161, 217, 155],
    [116, 196, 118], [49, 163, 84], [0, 109, 44],
], dtype="uint8")

grid_ll = grid.to_crs(4326)[["n_types", "geometry"]]    # lonboard needs lon/lat coordinates
grid_layer = PolygonLayer.from_geopandas(
    grid_ll,
    get_fill_color=ramp[grid_ll["n_types"].clip(upper=6)],    # pick each cell's color by its count
    opacity=0.4,
)
grid_map = Map(grid_layer)
legend([(f"{n}", ramp[n]) for n in (0, 1, 2, 3, 4, 5, 6)])
grid_map

### ✏️ Try it Out: gaps and the surface

Look at the preschool gaps map from Part 2 and the accessibility surface above. Which parts of San Francisco have the lowest access, and do the two maps agree?

*Your answer here.*

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part 4: pydeck or lonboard, at scale?

So far both libraries handled our data easily. To see where they differ, we need a lot more points. We take **every** place we pulled for San Francisco (offices and services too) and repeat it 25 times, about 1.8 million points. We build one map in each library, and time how long each takes to build and export.

In [ ]:
# every place, repeated 25 times to make a big dataset
big = pd.concat([places[["geometry"]]] * 25, ignore_index=True)
print(len(big), "points")

# pydeck: needs lon/lat columns, then build and export
t0 = time.perf_counter()
df_big = pd.DataFrame({"lon": big.geometry.x, "lat": big.geometry.y})
big_pydeck = pdk.Deck(
    layers=[pdk.Layer("ScatterplotLayer", df_big, get_position=["lon", "lat"], get_fill_color=[230, 90, 40], get_radius=15)],
    initial_view_state=pdk.ViewState(latitude=37.77, longitude=-122.43, zoom=11),
)
big_pydeck.to_html("scale_pydeck.html")
t_pydeck = time.perf_counter() - t0

In [ ]:
# lonboard: pass the GeoDataFrame directly, then build and export
t0 = time.perf_counter()
big_lonboard = Map(ScatterplotLayer.from_geopandas(big, get_fill_color=[230, 90, 40], radius_min_pixels=2))
big_lonboard.to_html("scale_lonboard.html")
t_lonboard = time.perf_counter() - t0

In [ ]:
# compare build time and file size
for name, t, f in [("pydeck", t_pydeck, "scale_pydeck.html"), ("lonboard", t_lonboard, "scale_lonboard.html")]:
    print(f"{name}: {t:.1f} s to build and export, {os.path.getsize(f) / 1e6:.1f} MB")



Open both files (right-click in the file browser, then **Open in New Browser Tab**) and pan and zoom around. Which one feels smoother?

### ✏️ Try it Out: pydeck or lonboard?

Using what you saw in the scale demo, in 2 to 3 sentences: if you were making a map for your final project with hundreds of thousands of data points, which library would you use, and why?

*Your answer here.*

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit.

Upload your submission zip to **LAB11 - Accessibility & Publishing the Interactive Map** on [Gradescope](https://www.gradescope.com/).

In [ ]:
grader.export(pdf=False, force_save=True, run_tests=True)